# Hybrid State-Space Attention (HSA) - Honest Benchmark
**No cherry-picking. Run it yourself and draw your own conclusions.**

Author: Vladimir0-1 | License: MIT

In [ ]:
!pip install torch transformers numpy matplotlib datasets tqdm -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer

# Clone repo (skip if already cloned)
!git clone https://github.com/Vladimir0-1/Hybrid-State-Space-Attention-HSA-.git 2>/dev/null || echo "Repo already exists"
%cd Hybrid-State-Space-Attention-HSA-

from hsa import HybridStateSpaceAttention

## 1. Define Models for Fair Comparison

In [ ]:
class StandardAttention(nn.Module):
    """Standard multi-head attention (baseline)"""
    def __init__(self, hidden_size, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(hidden_size, hidden_size * 3)
        self.proj = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(x)


class TinyTransformer(nn.Module):
    """Minimal transformer for fair comparison"""
    def __init__(self, vocab_size, hidden_size, num_heads, num_layers, attention_class, **attn_kwargs):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'norm1': nn.LayerNorm(hidden_size),
                'attn': attention_class(hidden_size, num_heads, **attn_kwargs),
                'norm2': nn.LayerNorm(hidden_size),
                'ffn': nn.Sequential(
                    nn.Linear(hidden_size, hidden_size * 4),
                    nn.GELU(),
                    nn.Linear(hidden_size * 4, hidden_size)
                )
            }) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            residual = x
            x = layer['norm1'](x)
            x = layer['attn'](x)
            x = residual + x
            residual = x
            x = layer['norm2'](x)
            x = layer['ffn'](x)
            x = residual + x
        x = self.norm(x)
        return self.lm_head(x)

## 2. Speed & Memory Benchmark (Honest)

In [ ]:
def benchmark_speed_memory(model, seq_len, device='cuda', num_iters=20):
    """Measure forward + backward time and peak memory"""
    model = model.to(device)
    model.train()
    x = torch.randint(0, 1000, (1, seq_len)).to(device)
    
    # Warmup
    for _ in range(5):
        out = model(x)
        loss = out.mean()
        loss.backward()
        model.zero_grad()
    
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    
    times = []
    for _ in range(num_iters):
        start = time.time()
        out = model(x)
        loss = out.mean()
        loss.backward()
        model.zero_grad()
        if device == 'cuda':
            torch.cuda.synchronize()
        times.append(time.time() - start)
    
    peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if device == 'cuda' else None
    
    return {
        'mean_time_ms': np.mean(times) * 1000,
        'std_time_ms': np.std(times) * 1000,
        'peak_memory_mb': peak_mem
    }


seq_lengths = [128, 256, 512, 1024, 2048, 4096]
configs = {
    'hidden_size': 256,
    'num_heads': 8,
    'num_layers': 4,
    'vocab_size': 10000
}

results = {'standard': [], 'hsa': []}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

for seq_len in tqdm(seq_lengths, desc="Benchmarking"):
    # Standard model
    model_std = TinyTransformer(
        **configs,
        attention_class=StandardAttention,
        dropout=0.1
    )
    std_res = benchmark_speed_memory(model_std, seq_len, device)
    
    # HSA model
    model_hsa = TinyTransformer(
        **configs,
        attention_class=HybridStateSpaceAttention,
        window_size=512,
        num_global_tokens=64
    )
    hsa_res = benchmark_speed_memory(model_hsa, seq_len, device)
    
    results['standard'].append(std_res)
    results['hsa'].append(hsa_res)

## 3. Plot Results (No Commentary, Just Data)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Speed plot
ax = axes[0]
std_times = [r['mean_time_ms'] for r in results['standard']]
hsa_times = [r['mean_time_ms'] for r in results['hsa']]
std_err = [r['std_time_ms'] for r in results['standard']]
hsa_err = [r['std_time_ms'] for r in results['hsa']]

ax.errorbar(seq_lengths, std_times, yerr=std_err, fmt='o-', label='Standard', capsize=5)
ax.errorbar(seq_lengths, hsa_times, yerr=hsa_err, fmt='s-', label='HSA', capsize=5)
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Time (ms)')
ax.set_title('Forward+Backward Time (lower is better)')
ax.legend()
ax.grid(True, alpha=0.3)

# Memory plot
if device == 'cuda':
    ax = axes[1]
    std_mem = [r['peak_memory_mb'] for r in results['standard']]
    hsa_mem = [r['peak_memory_mb'] for r in results['hsa']]
    ax.plot(seq_lengths, std_mem, 'o-', label='Standard')
    ax.plot(seq_lengths, hsa_mem, 's-', label='HSA')
    ax.set_xlabel('Sequence Length')
    ax.set_ylabel('Peak Memory (MB)')
    ax.set_title('GPU Memory Usage (lower is better)')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Memory benchmark requires CUDA', ha='center')
    axes[1].set_title('Memory (unavailable)')

plt.tight_layout()
plt.savefig('honest_benchmark.png', dpi=150)
plt.show()

# Print raw data
print("\n📊 Raw Benchmark Data:")
print("="*70)
print(f"{'Seq Len':>8} | {'Standard (ms)':>12} | {'HSA (ms)':>10} | {'Speedup':>7} | {'Std Mem (MB)':>11} | {'HSA Mem (MB)':>11}")
print("-"*70)
for i, seq in enumerate(seq_lengths):
    std_mem_str = f"{results['standard'][i]['peak_memory_mb']:.1f}" if results['standard'][i]['peak_memory_mb'] else "N/A"
    hsa_mem_str = f"{results['hsa'][i]['peak_memory_mb']:.1f}" if results['hsa'][i]['peak_memory_mb'] else "N/A"
    speedup = results['standard'][i]['mean_time_ms'] / results['hsa'][i]['mean_time_ms']
    print(f"{seq:8d} | {results['standard'][i]['mean_time_ms']:10.2f} ±{results['standard'][i]['std_time_ms']:.1f} | {results['hsa'][i]['mean_time_ms']:8.2f} ±{results['hsa'][i]['std_time_ms']:.1f} | {speedup:6.1f}x | {std_mem_str:>11} | {hsa_mem_str:>11}")

## 4. Quality Benchmark (Language Modeling Perplexity)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in dataloader:
        x = batch['input_ids'].to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), x.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


# Load tiny dataset
print("Loading WikiText-2 dataset...")
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)

print("\nTraining Standard Model...")
model_std = TinyTransformer(**configs, attention_class=StandardAttention).to(device)
opt_std = torch.optim.Adam(model_std.parameters(), lr=1e-3)
losses_std = []
for epoch in range(3):
    loss = train_one_epoch(model_std, dataloader, opt_std, device)
    losses_std.append(loss)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}")

print("\nTraining HSA Model...")
model_hsa = TinyTransformer(**configs, attention_class=HybridStateSpaceAttention, window_size=512, num_global_tokens=64).to(device)
opt_hsa = torch.optim.Adam(model_hsa.parameters(), lr=1e-3)
losses_hsa = []
for epoch in range(3):
    loss = train_one_epoch(model_hsa, dataloader, opt_hsa, device)
    losses_hsa.append(loss)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(range(1, 4), losses_std, 'o-', label='Standard')
plt.plot(range(1, 4), losses_hsa, 's-', label='HSA')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss (lower is better)')
plt.title('Training Convergence on WikiText-2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('convergence.png', dpi=150)
plt.show()

print(f"\nFinal Loss: Standard {losses_std[-1]:.4f} | HSA {losses_hsa[-1]:.4f}")
print(f"Difference: {(losses_hsa[-1] - losses_std[-1])*100:.2f}%")

## 5. Long Context Test (if enough memory)

In [ ]:
long_seq = 8192
if device == 'cuda' and torch.cuda.get_device_properties(0).total_memory > 8e9:
    print(f"Testing with sequence length {long_seq}...")
    
    # Standard (may OOM)
    try:
        model_std_long = TinyTransformer(**configs, attention_class=StandardAttention).to(device)
        x = torch.randint(0, 1000, (1, long_seq)).to(device)
        start = time.time()
        out = model_std_long(x)
        std_time = time.time() - start
        print(f"Standard: {std_time:.2f}s")
    except RuntimeError as e:
        print(f"Standard: OOM - {str(e)[:50]}")
        std_time = None
    
    # HSA
    model_hsa_long = TinyTransformer(**configs, attention_class=HybridStateSpaceAttention, window_size=1024, num_global_tokens=128).to(device)
    start = time.time()
    out = model_hsa_long(x)
    hsa_time = time.time() - start
    print(f"HSA: {hsa_time:.2f}s")
    
    if std_time:
        print(f"Speedup on {long_seq}: {std_time/hsa_time:.1f}x")
else:
    print(f"Skipping {long_seq} test (not enough GPU memory)")

## Observations (you draw your own conclusions)

**What you just measured:**
1. Speed (forward+backward) across different sequence lengths
2. GPU memory usage
3. Training convergence (perplexity)
4. Long context capability

**Interpretation is left to the reader.**

---
*Repository: https://github.com/Vladimir0-1/Hybrid-State-Space-Attention-HSA-*
*Run this notebook again to verify results on your own hardware.*